In [2]:
pip install google-adk

In [3]:
from google.adk.models import LlmRequest, LlmResponse
from google.adk.agents.callback_context import CallbackContext
from google.genai import types

In [4]:
from google.adk.tools import google_search, agent_tool
from google.adk.agents import LlmAgent, SequentialAgent

SEARCH_INSTRUCTIONS = """
You are the Search Agent. Your sole purpose is to gather raw information to answer the user's question.

Your tasks:
1. Analyze the user's question and identify the core intent and keywords.
2. Use the provided google_search tool to find the most accurate and up-to-date data.
3. Extract three specific components: the Answer, the Title of the page, and the Source URL.
4. Present this data objectively. Do not attempt to format it into a final response; simply provide the facts and the metadata (Title/Link) clearly.
"""

search_agent = LlmAgent(
    model="gemini-2.5-flash-lite",
    name="search_agent",
    description="",
    instruction=SEARCH_INSTRUCTIONS,
    tools=[google_search]
)

In [5]:
CRITIQUE_INSTRUCTIONS = """
You are the Critique Agent. Your role is to act as a rigorous editor for the Search Agent's output.

Your tasks:
1. Evaluate the Search Agent's response against the user's original question. Does it actually answer the question?
2. Check for technical requirements: Is there a Title? Is there a valid Link? Is the information clear?
3. Look for "hallucinations" or logical inconsistencies.
4. Provide a list of specific, actionable suggestions for improvement. If the answer is perfect, state that no changes are needed.
5. Focus on improving clarity, tone, and the transparency of the source.
"""

critique_agent = LlmAgent(
    model="gemini-2.5-flash-lite",
    name="critique_agent",
    description="Checks that the data retrieve from search is sufficient",
    instruction=CRITIQUE_INSTRUCTIONS
)

In [6]:
REFINE_INSTRUCTIONS = """
You are the Refine Agent. Your goal is to synthesize the Search Agent's data and the Critique Agent's feedback into a perfect final response.

Your tasks:
1. Rewrite the initial findings into a professional, engaging, and easy-to-read answer.
2. Address every suggestion made by the Critique Agent to ensure the highest quality.
3. You MUST format the final output as follows:
   - Provide the answer in one or two concise paragraphs.
   - Below the answer, include a section titled "Source".
   - Under "Source", list the Page Title and the clickable URL Link.
4. Ensure the tone is helpful and that the link is clearly visible.
"""

refine_agent = LlmAgent(
    model="gemini-2.5-flash-lite",
    name="refine_agent",
    description="Refines the output of the Search Agent",
    instruction=REFINE_INSTRUCTIONS
)

In [7]:
greeter_agent = LlmAgent(
    model="gemini-2.5-flash-lite",
    name="greeter_agent",
    description="Greets the user and asks for a question",
    instruction="""
    You are the Greeter Agent. Your sole purpose is to greet the user and ask them a question
    to begin the conversation."""
)

In [8]:
root_agent = SequentialAgent(
    name="root_agent",
    sub_agents=[greeter_agent, search_agent, critique_agent, refine_agent]
)

In [12]:
!gcloud config set project qwiklabs-gcp-02-e8fb21dc2037

Updated property [core/project].


In [13]:
!gsutil mb -l us-central1 gs://qwiklabs-gcp-02-e8fb21dc2037-staging/

Creating gs://qwiklabs-gcp-02-e8fb21dc2037-staging/...


In [15]:
from vertexai import agent_engines
from vertexai.preview import reasoning_engines
import vertexai

vertexai.init(
  project="qwiklabs-gcp-02-e8fb21dc2037",
  location="us-central1",
  staging_bucket="gs://qwiklabs-gcp-02-e8fb21dc2037-staging",
)

app = reasoning_engines.AdkApp(
    agent=root_agent
)

remote_agent = agent_engines.create(
  app,
  requirements=["google-cloud-aiplatform[agent_engines,adk]"],
)

INFO:vertexai.agent_engines:Identified the following requirements: {'google-cloud-aiplatform': '1.135.0', 'cloudpickle': '3.1.2', 'pydantic': '2.12.3'}
INFO:vertexai.agent_engines:The following requirements are appended: {'pydantic==2.12.3', 'cloudpickle==3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'pydantic==2.12.3', 'cloudpickle==3.1.2']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-02-e8fb21dc2037-staging
INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-02-e8fb21dc2037-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-02-e8fb21dc2037-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-02-e8fb21dc2037-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backin

In [16]:
for event in remote_agent.stream_query(
  user_id="agent-engine-test-user",
  message="Give me the news highlights in the world of sports.",
):
  print(event)

{'model_version': 'gemini-2.5-flash-lite', 'content': {'parts': [{'text': "Hello there! I'm the Greeter Agent. What's on your mind today?"}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 19, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 19}], 'prompt_token_count': 72, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 72}], 'total_token_count': 91, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -0.18728922542772794, 'invocation_id': 'e-23ed9631-9471-46cd-8c12-9626fc7b0c72', 'author': 'greeter_agent', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'id': 'b8978446-0ad2-469e-9360-6d2045f55f3c', 'timestamp': 1772744678.875178}
{'model_version': 'gemini-2.5-flash-lite', 'content': {'parts': [{'text': ''}], 'role': 'model'}, 'grounding_metadata': {}, 'finish_reason': 'STOP', 'usage_metadata': {'prompt_token_count': 178, 'prompt_tokens_details': [{'